In [ ]:
import rasterio
import numpy as np
import pandas as pd
import glob
import os

MONTHLY_DIR = "/Users/cherryleheu/Documents/HCDP/Data/rf_all/monthly/"

def mm_to_inches(val):
    return val / 25.4 if val is not None and not np.isnan(val) else np.nan

records = []

for tif in sorted(glob.glob(os.path.join(MONTHLY_DIR, "rainfall_*_04.tif"))):
    year = int(os.path.basename(tif).split("_")[1])
    with rasterio.open(tif) as src:
        arr = src.read(1).astype(float)
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
        mean_inches = mm_to_inches(np.nanmean(arr))
    records.append({"year": year, "mean_inches": round(mean_inches, 2)})

df = pd.DataFrame(records)
df["rank"] = df["mean_inches"].rank(method="min", ascending=False).astype(int)
df = df.sort_values("rank").reset_index(drop=True)

display(df)